# Bronze Ingestion — CMS Source Files to Delta Tables

This notebook ingests the five CMS source CSV files from the `raw.landing` volume into bronze Delta tables. The bronze layer is a near 1:1 copy of source data, with two deliberate additions:

1. **Explicit schemas** declared for every column — no `inferSchema` — so CMS schema drift fails loudly instead of silently.
2. **Ingestion metadata** (`_ingested_at` timestamp, `_source_file` filename) added to every table for lineage.

Source files (in `/Volumes/medicare_provider_quality/raw/landing/`):
- `MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv` — Medicare physician utilization, ~9M rows
- `Hospital_General_Information.csv` — Hospital reference, ~5K rows
- `Complications_and_Deaths-Hospital.csv` — Hospital quality measures, mortality and complications
- `Unplanned_Hospital_Visits-Hospital.csv` — Hospital quality measures, readmissions
- `HCAHPS-Hospital.csv` — Hospital quality measures, patient experience surveys

Bronze tables will live in `medicare_provider_quality.bronze`.

In [0]:
# Setup: imports and constants

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, DateType, TimestampType
)

# Catalog and schema constants
CATALOG = "medicare_provider_quality"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/raw/landing"

# Set the current catalog and schema so we don't have to fully-qualify every table name
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"Catalog: {CATALOG}")
print(f"Schema:  {BRONZE_SCHEMA}")
print(f"Source volume: {VOLUME_PATH}")

Catalog: medicare_provider_quality
Schema:  bronze
Source volume: /Volumes/medicare_provider_quality/raw/landing


## 1. Hospital General Information

Source: `Hospital_General_Information.csv` (~5K rows, 1.4 MB)

Reference table for Medicare-certified hospitals. Maps CCN (CMS Certification Number) to hospital name, address, type, ownership, and overall star rating.

**Grain:** one row per hospital.

**Why ingest this first:** smallest file, simplest schema, fastest feedback loop for getting the bronze pattern right before scaling to the larger files.

In [0]:
# Ingest Hospital_General_Information.csv into bronze.hospital_info

source_file = f"{VOLUME_PATH}/Hospital_General_Information.csv"
target_table = "hospital_info"

# Read the CSV with inferred schema first — we'll inspect, then declare explicit types
df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")        # CMS files have embedded newlines in some address fields
    .option("escape", '"')              # standard CSV quote-escaping
    .csv(source_file)
)

# Show the inferred schema and a sample of the data
print(f"Source file: {source_file}")
print(f"Row count:   {df_raw.count():,}")
print(f"Column count: {len(df_raw.columns)}")
print("\n--- Schema (as read from CSV header) ---")
df_raw.printSchema()

Source file: /Volumes/medicare_provider_quality/raw/landing/Hospital_General_Information.csv
Row count:   5,432
Column count: 38

--- Schema (as read from CSV header) ---
root
 |-- Facility ID: string (nullable = true)
 |-- Facility Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City/Town: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP Code: string (nullable = true)
 |-- County/Parish: string (nullable = true)
 |-- Telephone Number: string (nullable = true)
 |-- Hospital Type: string (nullable = true)
 |-- Hospital Ownership: string (nullable = true)
 |-- Emergency Services: string (nullable = true)
 |-- Meets criteria for birthing friendly designation: string (nullable = true)
 |-- Hospital overall rating: string (nullable = true)
 |-- Hospital overall rating footnote: string (nullable = true)
 |-- MORT Group Measure Count: string (nullable = true)
 |-- Count of Facility MORT Measures: string (nullable = true)
 |-- Count of MORT Meas

In [0]:
# Bronze ingestion: hospital_info
# Explicit schema with snake_case column names.
# Numeric columns use try_cast (via SQL expression) to tolerate CMS sentinel
# strings like 'Not Available'.

hospital_info_schema = StructType([
    StructField("facility_id",                                  StringType(),  True),
    StructField("facility_name",                                StringType(),  True),
    StructField("address",                                      StringType(),  True),
    StructField("city_town",                                    StringType(),  True),
    StructField("state",                                        StringType(),  True),
    StructField("zip_code",                                     StringType(),  True),
    StructField("county_parish",                                StringType(),  True),
    StructField("telephone_number",                             StringType(),  True),
    StructField("hospital_type",                                StringType(),  True),
    StructField("hospital_ownership",                           StringType(),  True),
    StructField("emergency_services",                           StringType(),  True),
    StructField("meets_birthing_friendly_criteria",             StringType(),  True),
    StructField("hospital_overall_rating",                      IntegerType(), True),
    StructField("hospital_overall_rating_footnote",             StringType(),  True),
    StructField("mort_group_measure_count",                     IntegerType(), True),
    StructField("count_facility_mort_measures",                 IntegerType(), True),
    StructField("count_mort_measures_better",                   IntegerType(), True),
    StructField("count_mort_measures_no_different",             IntegerType(), True),
    StructField("count_mort_measures_worse",                    IntegerType(), True),
    StructField("mort_group_footnote",                          StringType(),  True),
    StructField("safety_group_measure_count",                   IntegerType(), True),
    StructField("count_facility_safety_measures",               IntegerType(), True),
    StructField("count_safety_measures_better",                 IntegerType(), True),
    StructField("count_safety_measures_no_different",           IntegerType(), True),
    StructField("count_safety_measures_worse",                  IntegerType(), True),
    StructField("safety_group_footnote",                        StringType(),  True),
    StructField("readm_group_measure_count",                    IntegerType(), True),
    StructField("count_facility_readm_measures",                IntegerType(), True),
    StructField("count_readm_measures_better",                  IntegerType(), True),
    StructField("count_readm_measures_no_different",            IntegerType(), True),
    StructField("count_readm_measures_worse",                   IntegerType(), True),
    StructField("readm_group_footnote",                         StringType(),  True),
    StructField("pt_exp_group_measure_count",                   IntegerType(), True),
    StructField("count_facility_pt_exp_measures",               IntegerType(), True),
    StructField("pt_exp_group_footnote",                        StringType(),  True),
    StructField("te_group_measure_count",                       IntegerType(), True),
    StructField("count_facility_te_measures",                   IntegerType(), True),
    StructField("te_group_footnote",                            StringType(),  True),
])

# Read raw CSV with original CMS column names
df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("nullValue", "")
    .csv(source_file)
)

# Rename columns positionally from CMS originals to snake_case
df_hospital_info = df_raw.toDF(*[f.name for f in hospital_info_schema.fields])

# Cast each column to its declared type
# String -> string is a no-op; numeric types use try_cast so CMS sentinel
# strings ('Not Available', etc.) become null rather than failing the load
for f_def in hospital_info_schema.fields:
    if isinstance(f_def.dataType, StringType):
        df_hospital_info = df_hospital_info.withColumn(
            f_def.name,
            F.col(f_def.name).cast(f_def.dataType)
        )
    else:
        df_hospital_info = df_hospital_info.withColumn(
            f_def.name,
            F.expr(f"try_cast(`{f_def.name}` AS {f_def.dataType.simpleString()})")
        )

# Add lineage columns
df_hospital_info = (
    df_hospital_info
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("Hospital_General_Information.csv"))
)

# Write as Delta, idempotent
(
    df_hospital_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {df_hospital_info.count():,} rows to {CATALOG}.{BRONZE_SCHEMA}.{target_table}")

Wrote 5,432 rows to medicare_provider_quality.bronze.hospital_info


In [0]:
# Verification: does the table exist and have data?
result = spark.sql(f"SELECT COUNT(*) AS row_count FROM {CATALOG}.{BRONZE_SCHEMA}.hospital_info")
result.show()

# Show first 3 rows to confirm data shape
spark.sql(f"SELECT facility_id, facility_name, state, hospital_overall_rating, _ingested_at FROM {CATALOG}.{BRONZE_SCHEMA}.hospital_info LIMIT 3").show(truncate=False)

+---------+
|row_count|
+---------+
|     5432|
+---------+

+-----------+-------------------------------+-----+-----------------------+-------------------------+
|facility_id|facility_name                  |state|hospital_overall_rating|_ingested_at             |
+-----------+-------------------------------+-----+-----------------------+-------------------------+
|010001     |SOUTHEAST HEALTH MEDICAL CENTER|AL   |4                      |2026-05-26 14:13:06.38964|
|010005     |MARSHALL MEDICAL CENTERS       |AL   |3                      |2026-05-26 14:13:06.38964|
|010006     |NORTH ALABAMA MEDICAL CENTER   |AL   |2                      |2026-05-26 14:13:06.38964|
+-----------+-------------------------------+-----+-----------------------+-------------------------+



## 2. Complications and Deaths - Hospital

Source: `Complications_and_Deaths-Hospital.csv` (~150K rows, 22 MB)

Hospital-level quality measures covering 30-day mortality (heart attack, heart failure, pneumonia, COPD, stroke, CABG), hip/knee complications, and CMS Patient Safety Indicators.

**Grain:** one row per hospital per measure.

**Key columns we'll use in silver:** `facility_id` (joins to hospital_info), `measure_id` (the measure code, e.g. `MORT_30_AMI`), `score` (the measured rate).

In [0]:
# Inspect Complications_and_Deaths-Hospital.csv before declaring the schema

source_file = f"{VOLUME_PATH}/Complications_and_Deaths-Hospital.csv"
target_table = "complications_and_deaths"

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(source_file)
)

print(f"Source file:  {source_file}")
print(f"Row count:    {df_raw.count():,}")
print(f"Column count: {len(df_raw.columns)}")
print("\n--- Columns ---")
for col_name in df_raw.columns:
    print(f"  {col_name}")

Source file:  /Volumes/medicare_provider_quality/raw/landing/Complications_and_Deaths-Hospital.csv
Row count:    95,840
Column count: 18

--- Columns ---
  Facility ID
  Facility Name
  Address
  City/Town
  State
  ZIP Code
  County/Parish
  Telephone Number
  Measure ID
  Measure Name
  Compared to National
  Denominator
  Score
  Lower Estimate
  Higher Estimate
  Footnote
  Start Date
  End Date


In [0]:
# Bronze ingestion: complications_and_deaths
# Per-hospital, per-measure quality data with mortality and complication rates

complications_schema = StructType([
    StructField("facility_id",          StringType(), True),
    StructField("facility_name",        StringType(), True),
    StructField("address",              StringType(), True),
    StructField("city_town",            StringType(), True),
    StructField("state",                StringType(), True),
    StructField("zip_code",             StringType(), True),
    StructField("county_parish",        StringType(), True),
    StructField("telephone_number",     StringType(), True),
    StructField("measure_id",           StringType(), True),
    StructField("measure_name",         StringType(), True),
    StructField("compared_to_national", StringType(), True),
    StructField("denominator",          DoubleType(), True),
    StructField("score",                DoubleType(), True),
    StructField("lower_estimate",       DoubleType(), True),
    StructField("higher_estimate",      DoubleType(), True),
    StructField("footnote",             StringType(), True),
    StructField("start_date",           DateType(),   True),
    StructField("end_date",             DateType(),   True),
])

# Read raw CSV (original CMS column names)
df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("nullValue", "")
    .csv(source_file)
)

# Positional rename to snake_case
df = df_raw.toDF(*[f.name for f in complications_schema.fields])

# Cast each column per its declared type
# Dates need explicit format because CMS uses MM/dd/yyyy, not ISO
for f_def in complications_schema.fields:
    if isinstance(f_def.dataType, StringType):
        df = df.withColumn(f_def.name, F.col(f_def.name).cast(f_def.dataType))
    elif isinstance(f_def.dataType, DateType):
        df = df.withColumn(f_def.name, F.to_date(F.col(f_def.name), "MM/dd/yyyy"))
    else:
        df = df.withColumn(
            f_def.name,
            F.expr(f"try_cast(`{f_def.name}` AS {f_def.dataType.simpleString()})")
        )

# Lineage columns
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("Complications_and_Deaths-Hospital.csv"))
)

# Write as Delta
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {df.count():,} rows to {CATALOG}.{BRONZE_SCHEMA}.{target_table}")

Wrote 95,840 rows to medicare_provider_quality.bronze.complications_and_deaths


In [0]:
# Verify the table: row count + sample
spark.sql(f"SELECT COUNT(*) AS row_count FROM {CATALOG}.{BRONZE_SCHEMA}.complications_and_deaths").show()

spark.sql(f"""
    SELECT facility_id, measure_id, score, start_date, end_date
    FROM {CATALOG}.{BRONZE_SCHEMA}.complications_and_deaths
    WHERE measure_id = 'MORT_30_AMI'
    LIMIT 3
""").show(truncate=False)

+---------+
|row_count|
+---------+
|    95840|
+---------+

+-----------+-----------+-----+----------+----------+
|facility_id|measure_id |score|start_date|end_date  |
+-----------+-----------+-----+----------+----------+
|010001     |MORT_30_AMI|11.4 |2021-07-01|2024-06-30|
|010005     |MORT_30_AMI|NULL |2021-07-01|2024-06-30|
|010006     |MORT_30_AMI|14.5 |2021-07-01|2024-06-30|
+-----------+-----------+-----+----------+----------+



## 3. Unplanned Hospital Visits - Hospital

Source: `Unplanned_Hospital_Visits-Hospital.csv` (~125K rows, 18 MB)

Hospital-level readmission and unplanned-visit measures. Includes hospital-wide all-cause 30-day readmission and condition-specific readmission rates (heart attack, heart failure, pneumonia).

**Grain:** one row per hospital per measure.

**Structurally similar to Complications and Deaths** — same identification block plus measure data, but with two extra columns (`Number of Patients`, `Number of Patients Returned`) capturing the underlying counts behind the rate.

In [0]:
# Bronze ingestion: unplanned_hospital_visits

source_file = f"{VOLUME_PATH}/Unplanned_Hospital_Visits-Hospital.csv"
target_table = "unplanned_hospital_visits"

unplanned_schema = StructType([
    StructField("facility_id",                  StringType(), True),
    StructField("facility_name",                StringType(), True),
    StructField("address",                      StringType(), True),
    StructField("city_town",                    StringType(), True),
    StructField("state",                        StringType(), True),
    StructField("zip_code",                     StringType(), True),
    StructField("county_parish",                StringType(), True),
    StructField("telephone_number",             StringType(), True),
    StructField("measure_id",                   StringType(), True),
    StructField("measure_name",                 StringType(), True),
    StructField("compared_to_national",         StringType(), True),
    StructField("denominator",                  DoubleType(), True),
    StructField("score",                        DoubleType(), True),
    StructField("lower_estimate",               DoubleType(), True),
    StructField("higher_estimate",              DoubleType(), True),
    StructField("number_of_patients",           IntegerType(), True),
    StructField("number_of_patients_returned",  IntegerType(), True),
    StructField("footnote",                     StringType(), True),
    StructField("start_date",                   DateType(),   True),
    StructField("end_date",                     DateType(),   True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("nullValue", "")
    .csv(source_file)
)

df = df_raw.toDF(*[f.name for f in unplanned_schema.fields])

for f_def in unplanned_schema.fields:
    if isinstance(f_def.dataType, StringType):
        df = df.withColumn(f_def.name, F.col(f_def.name).cast(f_def.dataType))
    elif isinstance(f_def.dataType, DateType):
        df = df.withColumn(f_def.name, F.to_date(F.col(f_def.name), "MM/dd/yyyy"))
    else:
        df = df.withColumn(
            f_def.name,
            F.expr(f"try_cast(`{f_def.name}` AS {f_def.dataType.simpleString()})")
        )

df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("Unplanned_Hospital_Visits-Hospital.csv"))
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {df.count():,} rows to {CATALOG}.{BRONZE_SCHEMA}.{target_table}")

Wrote 67,088 rows to medicare_provider_quality.bronze.unplanned_hospital_visits


In [0]:
spark.sql(f"""
    SELECT facility_id, measure_id, score, number_of_patients, start_date
    FROM {CATALOG}.{BRONZE_SCHEMA}.unplanned_hospital_visits
    WHERE measure_id = 'READM_30_HOSP_WIDE'
    LIMIT 3
""").show(truncate=False)

+-----------+----------+-----+------------------+----------+
|facility_id|measure_id|score|number_of_patients|start_date|
+-----------+----------+-----+------------------+----------+
+-----------+----------+-----+------------------+----------+



In [0]:
spark.sql(f"""
    SELECT DISTINCT measure_id, measure_name
    FROM {CATALOG}.{BRONZE_SCHEMA}.unplanned_hospital_visits
    ORDER BY measure_id
""").show(truncate=False)

+-----------------+---------------------------------------------------------------------------------------+
|measure_id       |measure_name                                                                           |
+-----------------+---------------------------------------------------------------------------------------+
|EDAC_30_AMI      |Hospital return days for heart attack patients                                         |
|EDAC_30_HF       |Hospital return days for heart failure patients                                        |
|EDAC_30_PN       |Hospital return days for pneumonia patients                                            |
|Hybrid_HWR       |Hybrid Hospital-Wide All-Cause Readmission Measure (HWR)                               |
|OP_32            |Rate of unplanned hospital visits after colonoscopy (per 1,000 colonoscopies)          |
|OP_35_ADM        |Rate of inpatient admissions for patients receiving outpatient chemotherapy            |
|OP_35_ED         |Rate of e

In [0]:
spark.sql(f"""
    SELECT facility_id, measure_id, score, number_of_patients, start_date
    FROM {CATALOG}.{BRONZE_SCHEMA}.unplanned_hospital_visits
    WHERE measure_id = 'Hybrid_HWR'
    LIMIT 3
""").show(truncate=False)

+-----------+----------+-----+------------------+----------+
|facility_id|measure_id|score|number_of_patients|start_date|
+-----------+----------+-----+------------------+----------+
|010001     |Hybrid_HWR|15.1 |NULL              |2023-07-01|
|010005     |Hybrid_HWR|13.3 |NULL              |2023-07-01|
|010006     |Hybrid_HWR|15.9 |NULL              |2023-07-01|
+-----------+----------+-----+------------------+----------+



## 4. HCAHPS - Hospital (Patient Experience Survey)

Source: `HCAHPS-Hospital.csv` (~600K rows, 100 MB)

HCAHPS (Hospital Consumer Assessment of Healthcare Providers and Systems) is the national standardized patient experience survey. CMS publishes per-hospital results for ~35 survey questions covering nurse communication, doctor communication, hospital environment, pain management, discharge information, and overall hospital rating.

**Grain:** one row per hospital per HCAHPS measure ID per answer description.

The high row count comes from each survey question being broken out into multiple "answer percentage" rows (e.g., "Always", "Usually", "Sometimes or Never"). This is why the file is much larger than the other quality files despite covering the same ~5K hospitals.

In [0]:
# Inspect HCAHPS-Hospital.csv

source_file = f"{VOLUME_PATH}/HCAHPS-Hospital.csv"
target_table = "hcahps"

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(source_file)
)

print(f"Source file:  {source_file}")
print(f"Row count:    {df_raw.count():,}")
print(f"Column count: {len(df_raw.columns)}")
print("\n--- Columns ---")
for col_name in df_raw.columns:
    print(f"  {col_name}")

Source file:  /Volumes/medicare_provider_quality/raw/landing/HCAHPS-Hospital.csv
Row count:    325,856
Column count: 22

--- Columns ---
  Facility ID
  Facility Name
  Address
  City/Town
  State
  ZIP Code
  County/Parish
  Telephone Number
  HCAHPS Measure ID
  HCAHPS Question
  HCAHPS Answer Description
  Patient Survey Star Rating
  Patient Survey Star Rating Footnote
  HCAHPS Answer Percent
  HCAHPS Answer Percent Footnote
  HCAHPS Linear Mean Value
  Number of Completed Surveys
  Number of Completed Surveys Footnote
  Survey Response Rate Percent
  Survey Response Rate Percent Footnote
  Start Date
  End Date


In [0]:
# Bronze ingestion: hcahps

hcahps_schema = StructType([
    StructField("facility_id",                          StringType(),  True),
    StructField("facility_name",                        StringType(),  True),
    StructField("address",                              StringType(),  True),
    StructField("city_town",                            StringType(),  True),
    StructField("state",                                StringType(),  True),
    StructField("zip_code",                             StringType(),  True),
    StructField("county_parish",                        StringType(),  True),
    StructField("telephone_number",                     StringType(),  True),
    StructField("hcahps_measure_id",                    StringType(),  True),
    StructField("hcahps_question",                      StringType(),  True),
    StructField("hcahps_answer_description",            StringType(),  True),
    StructField("patient_survey_star_rating",           IntegerType(), True),
    StructField("patient_survey_star_rating_footnote",  StringType(),  True),
    StructField("hcahps_answer_percent",                IntegerType(), True),
    StructField("hcahps_answer_percent_footnote",       StringType(),  True),
    StructField("hcahps_linear_mean_value",             DoubleType(),  True),
    StructField("number_of_completed_surveys",          IntegerType(), True),
    StructField("number_of_completed_surveys_footnote", StringType(),  True),
    StructField("survey_response_rate_percent",         IntegerType(), True),
    StructField("survey_response_rate_percent_footnote",StringType(),  True),
    StructField("start_date",                           DateType(),    True),
    StructField("end_date",                             DateType(),    True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("nullValue", "")
    .csv(source_file)
)

df = df_raw.toDF(*[f.name for f in hcahps_schema.fields])

for f_def in hcahps_schema.fields:
    if isinstance(f_def.dataType, StringType):
        df = df.withColumn(f_def.name, F.col(f_def.name).cast(f_def.dataType))
    elif isinstance(f_def.dataType, DateType):
        df = df.withColumn(f_def.name, F.to_date(F.col(f_def.name), "MM/dd/yyyy"))
    else:
        df = df.withColumn(
            f_def.name,
            F.expr(f"try_cast(`{f_def.name}` AS {f_def.dataType.simpleString()})")
        )

df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("HCAHPS-Hospital.csv"))
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {df.count():,} rows to {CATALOG}.{BRONZE_SCHEMA}.{target_table}")

Wrote 325,856 rows to medicare_provider_quality.bronze.hcahps


In [0]:
spark.sql(f"""
    SELECT facility_id, hcahps_measure_id, hcahps_answer_description,
           hcahps_answer_percent, patient_survey_star_rating
    FROM {CATALOG}.{BRONZE_SCHEMA}.hcahps
    WHERE hcahps_measure_id = 'H_HSP_RATING_9_10'
      AND facility_id = '010001'
    LIMIT 3
""").show(truncate=False)

+-----------+-----------------+------------------------------------------------+---------------------+--------------------------+
|facility_id|hcahps_measure_id|hcahps_answer_description                       |hcahps_answer_percent|patient_survey_star_rating|
+-----------+-----------------+------------------------------------------------+---------------------+--------------------------+
|010001     |H_HSP_RATING_9_10|Patients who gave a rating of "9" or "10" (high)|73                   |NULL                      |
+-----------+-----------------+------------------------------------------------+---------------------+--------------------------+



## 5. Medicare Physician & Other Practitioners — by Provider and Service (MUP-PHY)

Source: `MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv` (~9M rows, 2.85 GB)

The headline dataset — and the reason this project uses Spark. For each provider (identified by NPI), every HCPCS service code they billed Medicare for in calendar year 2023, with service counts, total Medicare allowed amount, and total Medicare payment.

**Grain:** one row per NPI × HCPCS service code × place of service.

**Why this file justifies the lakehouse:** at 9M rows it's past the comfort zone for pandas on a typical workstation. Reading from CSV will take several minutes; reading from Delta (silver, gold) will be near-instant. The format conversion alone is a real engineering win.

**Note on the schema:** the field reference for this dataset is published in the CMS MUP-PHY methodology document. The schema below uses the column names and types from that documentation directly.

In [0]:
# Quick schema inspection only (no count — that would take minutes on 9M rows)

source_file = f"{VOLUME_PATH}/MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv"
target_table = "medicare_physician_payments"

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(source_file)
)

print(f"Source file:  {source_file}")
print(f"Column count: {len(df_raw.columns)}")
print("\n--- Columns ---")
for col_name in df_raw.columns:
    print(f"  {col_name}")

Source file:  /Volumes/medicare_provider_quality/raw/landing/MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv
Column count: 28

--- Columns ---
  Rndrng_NPI
  Rndrng_Prvdr_Last_Org_Name
  Rndrng_Prvdr_First_Name
  Rndrng_Prvdr_MI
  Rndrng_Prvdr_Crdntls
  Rndrng_Prvdr_Ent_Cd
  Rndrng_Prvdr_St1
  Rndrng_Prvdr_St2
  Rndrng_Prvdr_City
  Rndrng_Prvdr_State_Abrvtn
  Rndrng_Prvdr_State_FIPS
  Rndrng_Prvdr_Zip5
  Rndrng_Prvdr_RUCA
  Rndrng_Prvdr_RUCA_Desc
  Rndrng_Prvdr_Cntry
  Rndrng_Prvdr_Type
  Rndrng_Prvdr_Mdcr_Prtcptg_Ind
  HCPCS_Cd
  HCPCS_Desc
  HCPCS_Drug_Ind
  Place_Of_Srvc
  Tot_Benes
  Tot_Srvcs
  Tot_Bene_Day_Srvcs
  Avg_Sbmtd_Chrg
  Avg_Mdcr_Alowd_Amt
  Avg_Mdcr_Pymt_Amt
  Avg_Mdcr_Stdzd_Amt


In [0]:
# Bronze ingestion: medicare_physician_payments
# The big one — 9M rows, ~2.85 GB CSV. Expect 2-5 minutes processing time.

mup_phy_schema = StructType([
    StructField("rndrng_npi",                       StringType(),  True),
    StructField("rndrng_prvdr_last_org_name",       StringType(),  True),
    StructField("rndrng_prvdr_first_name",          StringType(),  True),
    StructField("rndrng_prvdr_mi",                  StringType(),  True),
    StructField("rndrng_prvdr_crdntls",             StringType(),  True),
    StructField("rndrng_prvdr_ent_cd",              StringType(),  True),  # I=Individual, O=Organization
    StructField("rndrng_prvdr_st1",                 StringType(),  True),
    StructField("rndrng_prvdr_st2",                 StringType(),  True),
    StructField("rndrng_prvdr_city",                StringType(),  True),
    StructField("rndrng_prvdr_state_abrvtn",        StringType(),  True),
    StructField("rndrng_prvdr_state_fips",          StringType(),  True),
    StructField("rndrng_prvdr_zip5",                StringType(),  True),
    StructField("rndrng_prvdr_ruca",                StringType(),  True),
    StructField("rndrng_prvdr_ruca_desc",           StringType(),  True),
    StructField("rndrng_prvdr_cntry",               StringType(),  True),
    StructField("rndrng_prvdr_type",                StringType(),  True),  # specialty code
    StructField("rndrng_prvdr_mdcr_prtcptg_ind",    StringType(),  True),
    StructField("hcpcs_cd",                         StringType(),  True),
    StructField("hcpcs_desc",                       StringType(),  True),
    StructField("hcpcs_drug_ind",                   StringType(),  True),
    StructField("place_of_srvc",                    StringType(),  True),  # F=Facility, O=Non-facility
    StructField("tot_benes",                        IntegerType(), True),
    StructField("tot_srvcs",                        DoubleType(),  True),  # can be fractional
    StructField("tot_bene_day_srvcs",               IntegerType(), True),
    StructField("avg_sbmtd_chrg",                   DoubleType(),  True),
    StructField("avg_mdcr_alowd_amt",               DoubleType(),  True),
    StructField("avg_mdcr_pymt_amt",                DoubleType(),  True),
    StructField("avg_mdcr_stdzd_amt",               DoubleType(),  True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("nullValue", "")
    .csv(source_file)
)

# Positional rename to lowercase snake_case
df = df_raw.toDF(*[f.name for f in mup_phy_schema.fields])

# Cast each column. No date columns in this file, so just two branches.
for f_def in mup_phy_schema.fields:
    if isinstance(f_def.dataType, StringType):
        df = df.withColumn(f_def.name, F.col(f_def.name).cast(f_def.dataType))
    else:
        df = df.withColumn(
            f_def.name,
            F.expr(f"try_cast(`{f_def.name}` AS {f_def.dataType.simpleString()})")
        )

# Lineage
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv"))
)

# Write as Delta — this is where the time goes
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {df.count():,} rows to {CATALOG}.{BRONZE_SCHEMA}.{target_table}")

Wrote 9,660,647 rows to medicare_provider_quality.bronze.medicare_physician_payments


In [0]:
spark.sql(f"""
    SELECT 
        rndrng_npi,
        rndrng_prvdr_last_org_name,
        rndrng_prvdr_type,
        hcpcs_cd,
        tot_srvcs,
        avg_mdcr_pymt_amt
    FROM {CATALOG}.{BRONZE_SCHEMA}.medicare_physician_payments
    WHERE rndrng_prvdr_state_abrvtn = 'VA'
    ORDER BY avg_mdcr_pymt_amt DESC NULLS LAST
    LIMIT 5
""").show(truncate=False)

+----------+---------------------------------------------------------+--------------------------+--------+---------+-----------------+
|rndrng_npi|rndrng_prvdr_last_org_name                               |rndrng_prvdr_type         |hcpcs_cd|tot_srvcs|avg_mdcr_pymt_amt|
+----------+---------------------------------------------------------+--------------------------+--------+---------+-----------------+
|1528077658|Prince William Ambulatory Surgery Center, Llc            |Ambulatory Surgical Center|63685   |13.0     |19768.96         |
|1952749491|Haymarket Surgery Center, Llc                            |Ambulatory Surgical Center|63685   |12.0     |19768.96         |
|1124777321|Virginia Cardiovascular Specialists                      |Ambulatory Surgical Center|33264   |22.0     |19393.14         |
|1124777321|Virginia Cardiovascular Specialists                      |Ambulatory Surgical Center|33249   |24.0     |19385.75         |
|1063058303|Chesapeake Regional Surgery Center At Virgi

In [0]:
spark.sql(f"""
    SELECT DISTINCT rndrng_prvdr_type
    FROM {CATALOG}.{BRONZE_SCHEMA}.medicare_physician_payments
    ORDER BY rndrng_prvdr_type
    LIMIT 20
""").show(truncate=False)

+------------------------------------------------+
|rndrng_prvdr_type                               |
+------------------------------------------------+
|Addiction Medicine                              |
|Adult Congenital Heart Disease                  |
|Advanced Heart Failure and Transplant Cardiology|
|All Other Suppliers                             |
|Allergy/ Immunology                             |
|Ambulance Service Provider                      |
|Ambulatory Surgical Center                      |
|Anesthesiology                                  |
|Anesthesiology Assistant                        |
|Audiologist                                     |
|Cardiac Surgery                                 |
|Cardiology                                      |
|Centralized Flu                                 |
|Certified Clinical Nurse Specialist             |
|Certified Nurse Midwife                         |
|Certified Registered Nurse Anesthetist (CRNA)   |
|Chiropractic                  